# 🌧️ CNN-LSTM Rainfall Prediction - Google Colab Pipeline

Complete pipeline for training and evaluating rainfall prediction model on Google Colab with support for 4 split ratios and 8 classification metrics.

**Features:**
- ✓ Automatic GPU/CPU detection
- ✓ Data loading and preprocessing
- ✓ Training with validation
- ✓ Evaluation on 4 split ratios (60:40, 70:30, 80:20, 90:10)
- ✓ 8 classification metrics (Accuracy, F1, Precision, Recall, Sensitivity, AUC, Error Rate, Confusion Matrix)
- ✓ Results visualization

**Runtime:** ~25-35 minutes with GPU, ~45-60 minutes with CPU

## Step 1: Install Dependencies & Clone Repository

In [ ]:
!pip install -q torch scikit-learn pandas numpy matplotlib

!git clone https://github.com/Rishabh5649/Rainfall_Anomaly_Detection.git /content/cnn_lstm_rainfall_system

print("[✓] Installation complete!")
print("[✓] Repository cloned!")

## Step 2: Setup Environment & Verify Installation

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

# Setup paths
os.chdir('/content/cnn_lstm_rainfall_system')
sys.path.insert(0, '/content/cnn_lstm_rainfall_system')

# Detect environment
try:
    import google.colab
    print("[✓] Running on Google Colab")
except ImportError:
    print("[·] Running locally")

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Device: {device}")
if torch.cuda.is_available():
    print(f"[✓] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[·] Using CPU (GPU not available)")

print(f"\n[✓] PyTorch version: {torch.__version__}")
print(f"[✓] Setup complete!")

## Step 3: Load & Test Data Pipeline

In [ ]:
from colab_data_pipeline import build_dataset

print("="*80)
print("  LOADING DATA WITH 80:20 SPLIT")
print("="*80)

X_train, y_train, X_val, y_val, X_test, y_test, scaler, subdivisions = \
    build_dataset(seq_len=24, train_test_ratio="80:20", fit_new_scaler=False)

print(f"\n[✓] Data loaded successfully!")
print(f"\n    Training:   X={X_train.shape}, y={y_train.shape}")
print(f"    Validation: X={X_val.shape}, y={y_val.shape}")
print(f"    Test:       X={X_test.shape}, y={y_test.shape}")
print(f"    Subdivisions: {len(subdivisions)}")
print(f"    Features: RAINFALL, ENSO, IOD, MONTH_SIN, MONTH_COS, YEAR_NORM")

## Step 4: Train Model (Optional - Skip if Model Already Trained)

In [ ]:
from colab_train import train

print("="*80)
print("  TRAINING MODEL WITH 80:20 SPLIT")
print("="*80)

# Train with custom parameters (comment out if you just want defaults)
model, history = train(
    epochs=50,              # Reduced from 80 for faster training
    batch_size=256,
    lr=3e-3,
    weight_decay=1e-4,
    patience=15,
    seq_len=24,
    split_ratio="80:20"
)

print(f"\n[✓] Training complete!")
print(f"[✓] Best model saved to: checkpoints/best_model.pt")

## Step 5: Evaluate on All 4 Split Ratios

In [ ]:
from colab_evaluate import evaluate_all_ratios

print("="*80)
print("  EVALUATING ON ALL 4 SPLIT RATIOS")
print("="*80)

results = evaluate_all_ratios(
    batch_size=512,
    split_ratios=["60:40", "70:30", "80:20", "90:10"]
)

print("\n[✓] Evaluation complete!")

## Step 6: View Results Summary

In [ ]:
with open('outputs/colab_metrics.json') as f:
    results = json.load(f)

print("\n" + "="*80)
print("RESULTS SUMMARY - ALL 4 SPLIT RATIOS (TEST SET)")
print("="*80)
print(f"{'Ratio':<12} {'Accuracy':<12} {'F1':<12} {'Precision':<12} {'Recall':<12} {'AUC':<10}")
print("-" * 80)

for ratio in ["60:40", "70:30", "80:20", "90:10"]:
    t = results['ratios'][ratio]['splits']['test']
    print(f"{ratio:<12} {t['accuracy']:<12.4f} {t['f1_score']:<12.4f} {t['precision']:<12.4f} {t['recall']:<12.4f} {t['auc']:<10.4f}")

# Detailed metrics for 80:20 split
print("\n" + "="*80)
print("DETAILED METRICS - 80:20 SPLIT (TEST SET)")
print("="*80)
test_80 = results['ratios']['80:20']['splits']['test']
print(f"✓ Accuracy:    {test_80['accuracy']:.4f} ({test_80['accuracy']*100:.2f}%)")
print(f"✓ Precision:   {test_80['precision']:.4f}")
print(f"✓ Recall:      {test_80['recall']:.4f}")
print(f"✓ Sensitivity: {test_80['sensitivity']:.4f}")
print(f"✓ F1-Score:    {test_80['f1_score']:.4f}")
print(f"✓ AUC:         {test_80['auc']:.4f}")
print(f"✓ Error Rate:  {test_80['error_rate']:.4f} ({test_80['error_rate']*100:.2f}%)")

print(f"\n✓ Confusion Matrix (Test):")
cm = test_80['confusion_matrix']
print(f"              Predicted Neg  Predicted Pos")
print(f"Actual Neg         {cm[0][0]:<14} {cm[0][1]}")
print(f"Actual Pos         {cm[1][0]:<14} {cm[1][1]}")

## Step 7: Visualize Metrics Comparison

In [ ]:
ratios = ["60:40", "70:30", "80:20", "90:10"]
accuracies = [results['ratios'][r]['splits']['test']['accuracy'] for r in ratios]
f1_scores = [results['ratios'][r]['splits']['test']['f1_score'] for r in ratios]
aucs = [results['ratios'][r]['splits']['test']['auc'] for r in ratios]
precisions = [results['ratios'][r]['splits']['test']['precision'] for r in ratios]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
axes[0, 0].bar(ratios, accuracies, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[0, 0].set_ylabel('Accuracy', fontsize=11)
axes[0, 0].set_title('Accuracy by Split Ratio', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim([0.75, 0.85])
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

# F1-Score
axes[0, 1].bar(ratios, f1_scores, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[0, 1].set_ylabel('F1-Score', fontsize=11)
axes[0, 1].set_title('F1-Score by Split Ratio', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0.75, 0.85])
for i, v in enumerate(f1_scores):
    axes[0, 1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

# Precision
axes[1, 0].bar(ratios, precisions, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[1, 0].set_ylabel('Precision', fontsize=11)
axes[1, 0].set_title('Precision by Split Ratio', fontsize=12, fontweight='bold')
axes[1, 0].set_ylim([0.75, 0.85])
for i, v in enumerate(precisions):
    axes[1, 0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

# AUC
axes[1, 1].bar(ratios, aucs, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[1, 1].set_ylabel('AUC', fontsize=11)
axes[1, 1].set_title('AUC by Split Ratio', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0.80, 0.90])
for i, v in enumerate(aucs):
    axes[1, 1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('Classification Metrics Comparison - Test Set', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('outputs/colab_metrics_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("[✓] Visualization saved to outputs/colab_metrics_visualization.png")

## Step 8 (Optional): Save Results to Google Drive

In [ ]:
import shutil

try:
    # Save metrics
    shutil.copy(
        'outputs/colab_metrics.json',
        '/content/drive/My Drive/colab_metrics.json'
    )
    
    # Save visualization
    shutil.copy(
        'outputs/colab_metrics_visualization.png',
        '/content/drive/My Drive/colab_metrics_visualization.png'
    )
    
    # Save model
    shutil.copy(
        'checkpoints/best_model.pt',
        '/content/drive/My Drive/best_model.pt'
    )
    
    print("[✓] Files saved to Google Drive!")
    print("[✓] Check 'My Drive/colab_metrics.json' for results")
except:
    print("[!] Could not save to Google Drive (not mounted or doesn't exist)")
    print("[·] Results are available in /content/cnn_lstm_rainfall_system/outputs/")